# Notebook 03 — CP01: Differential Expression Analysis

**Module:** 20 — Capstone Projects  
**Tier:** 1 — Master  
**Notebook:** 3 of 12  
**Time estimate:** 90 minutes

> We implement the core statistical logic of DESeq2 in pure Python/NumPy/SciPy.
> This is not a complete reimplementation — it is a pedagogical reconstruction
> of the key steps that you can explain in an interview.

---
## Step 2 — DESeq2 Core Logic (Simplified)

**Step 1: Size factor normalization (median-of-ratios)**
For each gene g and sample j: `ratio_gj = count_gj / geometric_mean_g`
Size factor for sample j: `sf_j = median_g(ratio_gj)`
Normalized counts: `count_gj / sf_j`

**Step 2: Log2 fold change**
`log2FC_g = log2(mean_normalized_treated_g + pseudocount) - log2(mean_normalized_control_g + pseudocount)`

**Step 3: Wald test (approximation)**
For simplicity, we use a negative-binomial GLM approximation: log-transform +
t-test on log-counts (a common approximation for low n).
For n=3 per group, degrees of freedom = 4 (conservative).

**Step 4: Multiple testing correction**
Benjamini-Hochberg FDR correction: sorts p-values, applies adaptive threshold.
Standard cutoff: FDR < 0.05 and |log2FC| > 1.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

# ---- Regenerate data (same seed as NB02) ----
N_GENES   = 5000; N_SAMPLES = 6; N_DE = 300
mu_base    = rng.lognormal(3.0, 1.5, N_GENES)
dispersion = rng.exponential(0.1, N_GENES) + 0.01
r = 1.0 / dispersion[:, None]; p = 1.0 / (1.0 + dispersion[:, None] * mu_base[:, None])
counts = rng.negative_binomial(r, p, size=(N_GENES, N_SAMPLES)).astype(float)
de_idx  = rng.choice(N_GENES, N_DE, replace=False)
up_idx, dn_idx = de_idx[:N_DE//2], de_idx[N_DE//2:]
fold_changes = rng.uniform(1.5, 4.0, N_DE)
counts[up_idx, 3:] = (counts[up_idx, 3:] * fold_changes[:N_DE//2, None]).astype(int)
counts[dn_idx, 3:] = np.maximum(1, (counts[dn_idx, 3:] / fold_changes[N_DE//2:, None]).astype(int))
counts = np.clip(counts, 0, None)
gene_names = [f'GENE_{i:04d}' for i in range(N_GENES)]

# ---- Step 1: Median-of-ratios size factor normalization ----

def median_of_ratios_normalization(counts):
    """Compute DESeq2-style size factors using median-of-ratios."""
    # Geometric mean per gene (across samples)
    log_counts = np.log(counts + 1e-9)  # avoid log(0)
    log_geo_mean = log_counts.mean(axis=1)  # (n_genes,)
    # Ratio of each sample to geometric mean
    log_ratios = log_counts - log_geo_mean[:, None]  # (n_genes, n_samples)
    # Only use genes with non-zero counts in all samples
    mask = (counts > 0).all(axis=1)
    size_factors = np.exp(np.median(log_ratios[mask], axis=0))
    return size_factors

size_factors = median_of_ratios_normalization(counts)
norm_counts  = counts / size_factors[None, :]
print(f'Size factors: {size_factors.round(3)}')

# ---- Step 2: Log2 fold change ----
PSEUDOCOUNT = 1.0
log2_ctrl = np.log2(norm_counts[:, :3].mean(axis=1) + PSEUDOCOUNT)
log2_trt  = np.log2(norm_counts[:, 3:].mean(axis=1) + PSEUDOCOUNT)
log2fc    = log2_trt - log2_ctrl  # (n_genes,)
base_mean = norm_counts.mean(axis=1)

# ---- Step 3: t-test on log2-normalized counts (approximation) ----
log2_norm = np.log2(norm_counts + PSEUDOCOUNT)  # (n_genes, 6)
pvalues = np.array([stats.ttest_ind(log2_norm[g, 3:], log2_norm[g, :3]).pvalue
                     for g in range(N_GENES)])

# ---- Step 4: BH FDR correction ----
def bh_correction(pvalues):
    """Benjamini-Hochberg FDR correction."""
    n = len(pvalues)
    order = np.argsort(pvalues)
    ranks  = np.empty_like(order); ranks[order] = np.arange(1, n+1)
    adjusted = np.minimum(1.0, pvalues * n / ranks)
    # Enforce monotonicity (cumulative minimum from last to first)
    adj = adjusted[order]
    for i in range(n-2, -1, -1): adj[i] = min(adj[i], adj[i+1])
    adjusted[order] = adj
    return adjusted

padj = bh_correction(pvalues)

# ---- Results DataFrame ----
results = pd.DataFrame({
    'gene':       gene_names,
    'base_mean':  base_mean,
    'log2FC':     log2fc,
    'pvalue':     pvalues,
    'padj':       padj,
    'true_de':    np.isin(np.arange(N_GENES), de_idx),
}).set_index('gene')

results['significant'] = (results['padj'] < 0.05) & (results['log2FC'].abs() > 1)

# ---- Performance evaluation ----
TP = (results['significant'] & results['true_de']).sum()
FP = (results['significant'] & ~results['true_de']).sum()
FN = (~results['significant'] & results['true_de']).sum()
TN = (~results['significant'] & ~results['true_de']).sum()
precision = TP / max(TP + FP, 1)
recall    = TP / max(TP + FN, 1)
f1        = 2 * precision * recall / max(precision + recall, 1e-9)

print(f'\n=== DE Analysis Performance ===')
print(f'Significant genes at FDR<0.05 & |log2FC|>1: {results["significant"].sum()}')
print(f'True positives: {TP}/{N_DE} ({TP/N_DE:.0%} recall)')
print(f'False positives: {FP}')
print(f'Precision: {precision:.3f}, Recall: {recall:.3f}, F1: {f1:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel A: Volcano plot
ax = axes[0]
neg_log10p = -np.log10(results['padj'].clip(1e-30))
colors_v = np.where(results['significant'] & results['true_de'], '#e15759',
             np.where(results['significant'] & ~results['true_de'], '#f28e2b',
               np.where(~results['significant'] & results['true_de'], '#4e79a7', '#aaaaaa')))
ax.scatter(results['log2FC'], neg_log10p, c=colors_v, s=8, alpha=0.5, rasterized=True)
ax.axhline(-np.log10(0.05), color='grey', ls='--', lw=1, label='FDR = 0.05')
ax.axvline(-1, color='grey', ls=':', lw=1); ax.axvline(1, color='grey', ls=':', lw=1)
ax.set_xlabel('log₂ Fold Change (treated / control)')
ax.set_ylabel('-log₁₀(adjusted p-value)')
ax.set_title(f'A. Volcano plot\n(TP={TP}, FP={FP})')
legend_handles = [
    mpatches.Patch(color='#e15759', label=f'TP sig & true DE ({TP})'),
    mpatches.Patch(color='#f28e2b', label=f'FP sig & not DE ({FP})'),
    mpatches.Patch(color='#4e79a7', label=f'FN not sig & true DE ({FN})'),
    mpatches.Patch(color='#aaaaaa', label='TN not sig & not DE'),
]
ax.legend(handles=legend_handles, fontsize=7, loc='upper left')

# Panel B: MA plot
ax = axes[1]
log2_mean = np.log2(results['base_mean'] + 1)
ax.scatter(log2_mean, results['log2FC'], c=colors_v, s=8, alpha=0.4, rasterized=True)
ax.axhline(0, color='black', lw=1)
ax.axhline(1, color='grey', ls='--', lw=0.8)
ax.axhline(-1, color='grey', ls='--', lw=0.8)
ax.set_xlabel('log₂ Mean Expression')
ax.set_ylabel('log₂ Fold Change')
ax.set_title('B. MA plot\n(DE genes distributed across expression levels)')

# Panel C: Precision-Recall curve
ax = axes[2]
from sklearn.metrics import precision_recall_curve
prc_score = -results['padj'].values
true_labels = results['true_de'].values.astype(int)
try:
    from sklearn.metrics import precision_recall_curve
    prec, rec, _ = precision_recall_curve(true_labels, prc_score)
    ax.plot(rec, prec, color='#4e79a7', lw=2)
    ax.axhline(N_DE/N_GENES, color='grey', ls='--', lw=1, label='Random classifier')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'C. Precision-Recall curve\n(F1={f1:.3f})')
    ax.legend(fontsize=9)
except ImportError:
    ax.text(0.5, 0.5, f'scikit-learn not installed\nF1={f1:.3f}\nPrecision={precision:.3f}\nRecall={recall:.3f}',
              ha='center', va='center', transform=ax.transAxes, fontsize=11)
    ax.axis('off')

import matplotlib.patches as mpatches
plt.suptitle('Module 20 CP01: Differential Expression Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cp01_de_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 12 — Reflection

> *[Why is the t-test on log-counts only an approximation of DESeq2's Wald test?
> What does DESeq2 actually model that this approximation ignores?
> At what sample size (per group) does the approximation become adequate?]*

---
*Next: `04_cp01_visualization_interpretation.ipynb`*